# B3 · KNNImputer: measuring imputation when you know the true values

**Type**: Implement with scikit-learn · **Level**: ●●● challenge · **Time**: about 50 minutes · **extended path**

## Objective

Evaluate imputations (mean, median, `KNNImputer` with various `k` and weights) with a clean method: **erase known values** and measure the reconstruction error. Verify that without scaling the `KNNImputer` works worse and that a better imputation does not
automatically translate into a better classification.

## Dataset

`load_wine` standardized (`Z`, truth) with 15% of the cells made missing at random (`Zm`, mask `mask`). `X_raw` is the same matrix in the original units, with the same missing cells (`Xm`).

## Educational purpose

Know **how much** an imputer is worth, not just that it "works". The known-truth trick (masking values you know) is the standard for comparing imputations, and you will need it in the following modules.

## Task

1. Write `imputation_rmse(imputer)`: apply `fit_transform` to `Zm` and return the RMSE **only on the masked cells** with respect to `Z`.
2. Save `base_rmse` with `mean` and `median` (`SimpleImputer`), and `knn_rmse` (`pd.Series`) for `KNNImputer(n_neighbors=k)` with `k` in `[1, 2, 3, 5, 10, 15, 30, 50]`. Save `best_k`. Add `knn5_distance` (k=5, `weights="distance"`).
3. **Without scaling:** apply `KNNImputer(5)` to `Xm` (original units), convert the result to standardized units (`(R - X.mean(0)) / X.std(0)` with the statistics of the complete data) and save `rmse_without_scaling` on the masked cells.
4. **Downstream effect.** With a `Pipeline` `StandardScaler` → imputer → `LogisticRegression(max_iter=1000)` compute in cross-validation (`StratifiedKFold(5, shuffle=True, random_state=0)`) the accuracy on `Xm` for `mean` and for `KNNImputer(5)`: save `mean_acc`, `knn_acc`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_wine
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_wine(return_X_y=True)
mu, sd = X.mean(0), X.std(0)
Z = (X - mu) / sd                                            # standardized truth
mask = np.random.default_rng(0).random(Z.shape) < 0.15       # 15% of the cells missing at random
Zm = np.where(mask, np.nan, Z)
Xm = np.where(mask, np.nan, X)
print(f"{mask.mean():.1%} of cells masked, {mask.sum()} values to reconstruct")

## Your work

In [ ]:
def imputation_rmse(imputer):
    ...

In [ ]:
base_rmse = {"mean": None, "median": None}
knn_rmse = pd.Series(dtype=float)
best_k = None
knn5_distance = None

In [ ]:
rmse_without_scaling = None
mean_acc = None
knn_acc = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`R = imputer.fit_transform(Zm); np.sqrt(((R[mask] - Z[mask]) ** 2).mean())`. Without scaling: `R = KNNImputer(5).fit_transform(Xm); Rz = (R - mu) / sd`. Pipeline: `make_pipeline(StandardScaler(), SimpleImputer(), LogisticRegression(max_iter=1000))` (`StandardScaler` ignores the NaN when it computes mean and standard deviation).'):
    check(close(imputation_rmse(SimpleImputer()), base_rmse["mean"]) and 0.95 < base_rmse["mean"] < 1.15, "RMSE of the mean imputation: about 1 (the standard deviation of the data)", "on the masked cells only")
    check(list(knn_rmse.index) == [1, 2, 3, 5, 10, 15, 30, 50] and best_k == knn_rmse.idxmin(), "RMSE curve as a function of k and best k", "one imputer for every value of k")
    check(knn_rmse.min() < 0.85 * base_rmse["mean"] and knn_rmse[1] > knn_rmse.min() + 0.03, "the KNNImputer reduces the error by over 15% compared to the mean; with k = 1 it is much noisier", "the wine columns are correlated")
    check(3 <= best_k <= 30 and close(knn5_distance, imputation_rmse(KNNImputer(n_neighbors=5, weights="distance"))), "best k between 3 and 30; result with weights computed", "weights='distance'")
    check(rmse_without_scaling > knn_rmse[5] + 0.03, "without scaling the KNNImputer reconstructs worse (over 0.03 more RMSE)", "proline dominates the distance")
    check(abs(knn_acc - mean_acc) < 0.05 and knn_acc > 0.9, "the classification changes little (less than 5 points): the task is easy and the logistic is robust to missing values", "compare knn_acc and mean_acc")
done()

---
The commented solution is in `solutions/B_with_sklearn/B3_knn_imputer_with_known_truth.ipynb`. Open it after you pass the check, or after you have really tried.